# 🌶️ Prueba del detector de locotes en Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Legoshi7755/deteccion-locotes/blob/main/colab_prueba_locotes.ipynb)

Este notebook usa la GPU gratuita de Colab para probar el modelo `best_frutos_video_ampliado.pt` (YOLO de segmentación, clase `locote_rojo`) con los mismos umbrales y filtros que la aplicación WATSOCOTE.

| Parte | Qué hace | Necesita |
|---|---|---|
| **1. Fotos** | Detecta y segmenta locotes en fotos | Fotos tuyas |
| **2. Video** | Anota un video completo y lo descarga | Un video grabado |
| **3. En vivo** *(experimental)* | Corre la aplicación estéreo con los dos celulares | Los dos celulares transmitiendo |
| **4. Calibrar** | Calibra las dos cámaras después de mover o cambiar los soportes | Tablero ChArUco impreso y los dos celulares |

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. Después ejecuta las celdas en orden con ▶ (o `Shift+Enter`).

## 0. Preparación

In [ ]:
#@title Verificar la GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ Sin GPU: cambia el tipo de entorno a 'T4 GPU' (funciona igual en CPU, pero más lento)."

In [ ]:
#@title Descargar el repositorio e instalar dependencias
#@markdown `VERSION`: **main** es la versión estable. **modelo-tallo** es la del pull request: agrega el modelo de locote + tallo y el punto de agarre 3D.
#@markdown Si cambias de versión, vuelve a ejecutar esta celda y las siguientes.
VERSION = "main"  #@param ["main", "modelo-tallo"]
# Colab ya trae PyTorch con CUDA y OpenCV; solo falta Ultralytics.
%cd /content
!test -d deteccion-locotes || git clone -q --depth 1 https://github.com/Legoshi7755/deteccion-locotes
%cd /content/deteccion-locotes
!git fetch -q --depth 1 origin {VERSION} && git checkout -q -B {VERSION} FETCH_HEAD && echo "Versión: {VERSION}" && git log --oneline -1
!pip install -q ultralytics==8.4.191

In [ ]:
#@title Verificar la integridad de los archivos (SHA256)
import hashlib, json
from pathlib import Path
hashes = json.loads(Path("SHA256.json").read_text())
malos = [f for f, h in hashes.items() if not Path(f).is_file() or hashlib.sha256(Path(f).read_bytes()).hexdigest() != h]
print("✅ Los", len(hashes), "archivos coinciden con SHA256.json" if not malos else f"⚠️ No coinciden: {malos}")

In [ ]:
#@title Cargar el modelo
CONFIANZA = 0.40  #@param {type:"slider", min:0.05, max:0.95, step:0.05}
import shutil, sys
from pathlib import Path
REPO = Path("/content/deteccion-locotes")
if str(REPO / "fotos") not in sys.path:
    sys.path.insert(0, str(REPO / "fotos"))
# Si cambiaste de VERSION, olvidar el código cargado antes para usar el de la versión nueva.
for nombre in list(sys.modules):
    if (getattr(sys.modules[nombre], "__file__", None) or "").startswith(str(REPO)):
        del sys.modules[nombre]
import prueba_imagenes as pi
pi.cargar_modelo(conf=CONFIANZA)
TIENE_TALLO = (REPO / "fotos" / "probar_locote_tallo.py").exists()
print("Modelo de tallo disponible:", "sí" if TIENE_TALLO else "no (elige VERSION = modelo-tallo para usarlo)")

DRIVE_CALIBRACIONES = Path("/content/drive/MyDrive/watsocote_calibraciones")

def montar_drive():
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

def carpeta_sesion(nombre):
    return REPO / "fotos" / "sesiones" / nombre

def asegurar_sesion(nombre):
    # Usa la calibración de Colab o, si no está (Colab se reinició), la restaura desde Drive.
    local = carpeta_sesion(nombre)
    if (local / "calibracion_charuco.npz").exists():
        return local
    montar_drive()
    remoto = DRIVE_CALIBRACIONES / nombre
    if not (remoto / "calibracion_charuco.npz").exists():
        raise FileNotFoundError(f"No encontré la calibración '{nombre}' ni en Colab ni en tu Drive ({remoto}).")
    shutil.copytree(remoto, local, dirs_exist_ok=True)
    print("Calibración restaurada desde Drive:", remoto)
    return local

def ruta_drive(texto):
    # Acepta '/content/drive/MyDrive/Claude/Fotos', 'MyDrive/Claude/Fotos' o 'Claude/Fotos'.
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    texto = texto.strip().strip("/")
    for prefijo in ("content/drive/MyDrive/", "content/drive/", "MyDrive/", "Mi unidad/"):
        if texto.startswith(prefijo):
            texto = texto[len(prefijo):]
    ruta = Path("/content/drive/MyDrive") / texto
    if not ruta.exists():
        raise FileNotFoundError(f"No encontré '{texto}' en tu Drive. Escribe la ruta desde 'Mi unidad', "
                                f"por ejemplo: Claude/Fotos_locote")
    print("Usando:", ruta)
    return ruta

## 1. Probar con fotos

Sube una o varias fotos desde tu computadora o celular. Si prefieres usar fotos de tu Google Drive, escribe en `CARPETA_DRIVE` la ruta de la carpeta **desde "Mi unidad"**, por ejemplo `Claude/Fotos_locote` (también sirve la ruta completa `/content/drive/MyDrive/Claude/Fotos_locote`). Déjalo vacío para subir fotos desde tu equipo.

`USAR_SAM` refina el contorno con MobileSAM, como en la captura puntual de la aplicación (más preciso, más lento).

In [ ]:
#@title Subir y analizar fotos
CARPETA_DRIVE = "Claude/Fotos_locote"  #@param {type:"string"}
USAR_SAM = False  #@param {type:"boolean"}
#@markdown `DETECTAR_TALLO`: usa el modelo nuevo y pinta locote (rojo), tallo (cian), hojas (verde) y ramas (magenta). Requiere `VERSION = modelo-tallo` en la parte 0.
DETECTAR_TALLO = False  #@param {type:"boolean"}
import cv2, matplotlib.pyplot as plt
from pathlib import Path

entrada = Path("/content/entradas"); entrada.mkdir(exist_ok=True)
salida = Path("/content/resultados"); salida.mkdir(exist_ok=True)
if CARPETA_DRIVE:
    fotos = [p for p in sorted(ruta_drive(CARPETA_DRIVE).rglob("*")) if p.suffix.lower() in pi.EXT_IMAGEN]
else:
    from google.colab import files
    fotos = []
    for nombre, datos in files.upload().items():
        (entrada / nombre).write_bytes(datos)
        if Path(nombre).suffix.lower() in pi.EXT_IMAGEN:
            fotos.append(entrada / nombre)
print(f"{len(fotos)} foto(s)")
if not fotos:
    print("⚠️ No hay fotos .jpg/.png en esa carpeta.")

if DETECTAR_TALLO and not TIENE_TALLO:
    print("⚠️ Esta versión no tiene el modelo de tallo: elige VERSION = modelo-tallo en la parte 0 y vuelve a ejecutarla.")
    DETECTAR_TALLO = False
if DETECTAR_TALLO:
    import probar_locote_tallo as pt
    from ultralytics import YOLO
    pt.CONF_LOCOTE = CONFIANZA
    modelo_tallo = YOLO(str(pt.MODELO_TALLO)); modelo_hojas = YOLO(str(pt.MODELO_HOJAS))

for ruta in fotos:
    imagen = cv2.imread(str(ruta))
    if imagen is None:
        print(ruta.name, "no se pudo leer"); continue
    if DETECTAR_TALLO:
        r = modelo_tallo.predict(imagen, conf=min(pt.CONF_TALLO, CONFIANZA), imgsz=960, device=pt.DISPOSITIVO, verbose=False)[0]
        rh = modelo_hojas.predict(imagen, conf=pt.CONF_HOJA_RAMA, imgsz=640, device=pt.DISPOSITIVO,
                                  verbose=False, classes=[2, 3], max_det=400)[0]
        vista, tallos = pt.dibujar(imagen, r, rh)
        avisos = [" ".join(k for k in ("tapado", "rama_cerca") if t.get(k)) for t in tallos]
        titulo = f"{ruta.name}: {len(tallos)} tallo(s) " + " ".join(f"{t['conf']:.2f}" + (f" ({a})" if a else "") for t, a in zip(tallos, avisos))
    else:
        vista, dets = pi.detectar(imagen, sam=USAR_SAM)
        titulo = f"{ruta.name}: {len(dets)} locote(s) " + " ".join(f"{d['confianza']:.2f}" for d in dets)
    cv2.imwrite(str(salida / f"{ruta.stem}_anotado.jpg"), vista)
    plt.figure(figsize=(9, 9)); plt.imshow(cv2.cvtColor(vista, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(titulo); plt.show()

In [ ]:
#@title Descargar las fotos anotadas (zip)
!cd /content && zip -qr resultados_fotos.zip resultados -i '*.jpg'
from google.colab import files
files.download("/content/resultados_fotos.zip")

## 2. Probar con un video

Graba un video del cultivo con el celular y súbelo, o escribe su ruta en Drive desde "Mi unidad" (por ejemplo `Claude/videos/cultivo.mp4`). Con `CONFIRMAR_DOS_CUADROS` un locote solo se marca si aparece en dos cuadros seguidos, igual que la detección continua de la aplicación; desactívalo para ver todas las detecciones crudas.

`ANALIZAR_CADA` = 1 analiza todos los cuadros; 2 o 3 lo hace más rápido en videos largos.

In [ ]:
#@title Subir y anotar un video
RUTA_VIDEO_DRIVE = ""  #@param {type:"string"}
ANALIZAR_CADA = 1  #@param {type:"integer"}
CONFIRMAR_DOS_CUADROS = True  #@param {type:"boolean"}
MAX_SEGUNDOS = 60  #@param {type:"integer"}
from pathlib import Path
if RUTA_VIDEO_DRIVE:
    video = ruta_drive(RUTA_VIDEO_DRIVE)
else:
    from google.colab import files
    subido = files.upload()
    nombre = next(iter(subido))
    video = Path("/content/entradas") / nombre
    video.parent.mkdir(exist_ok=True); video.write_bytes(subido[nombre])
Path("/content/resultados").mkdir(exist_ok=True)
crudo = Path("/content/resultados") / f"{video.stem}_crudo.mp4"
anotado = Path("/content/resultados") / f"{video.stem}_anotado.mp4"
pi.procesar_video(video, crudo, cada=max(1, ANALIZAR_CADA), confirmar=CONFIRMAR_DOS_CUADROS,
                  max_segundos=MAX_SEGUNDOS or None)
# Convertir a H.264 para que se vea en el navegador
!ffmpeg -y -loglevel error -i "{crudo}" -c:v libx264 -pix_fmt yuv420p -crf 23 "{anotado}" && rm "{crudo}"
print("Listo:", anotado)

In [ ]:
#@title Ver el video anotado
from base64 import b64encode
from IPython.display import HTML, display
tam = anotado.stat().st_size / 1e6
if tam > 60:
    print(f"El video pesa {tam:.0f} MB; descárgalo con la celda siguiente.")
else:
    display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{b64encode(anotado.read_bytes()).decode()}" type="video/mp4"></video>'))

In [ ]:
#@title Descargar el video anotado
from google.colab import files
files.download(str(anotado))

## 3. Aplicación en vivo con los dos celulares *(experimental)*

Corre la aplicación completa (`fotos/yolo_segmentacion_video_ampliado.py`) en Colab con GPU. El flujo:

```
Celulares ──VDO.Ninja──▶ tu navegador (página de la app) ──cuadros──▶ servidor en Colab (YOLO en GPU)
```

1. Ejecuta la celda **Iniciar servidor** y espera a que diga que está listo.
2. En **cada celular**, abre su enlace de emisión (se imprimen abajo), permite la cámara trasera e inicia la transmisión. Mantén la pantalla encendida.
3. Ejecuta **Abrir la aplicación** y entra al enlace que aparece. Ábrelo **en Chrome con la misma cuenta de Google** con la que usas Colab.

⚠️ **Ten en cuenta:**
- Cada par de imágenes viaja por internet hasta Colab: la detección continua será más lenta que en una PC local, según tu conexión. En "Diagnóstico y configuración" de la app ves cuánto tarda cada paso.
- Si actualizaste el repositorio, vuelve a ejecutar la parte 0 y esta celda, y recarga la página de la app con `Ctrl+F5`.
- La calibración incluida solo sirve para el montaje con el que se obtuvo (posición, lente y zoom de los celulares).
- Colab apaga la máquina tras un rato de inactividad. Si activas "guardar datos de entrenamiento" en la app, las fotos quedan en `/content/deteccion-locotes/fotos/dataset_carrito_pendiente`, y **se pierden al cerrar Colab** salvo que las copies a Drive (celda al final).

In [ ]:
#@title Iniciar servidor
#@markdown `TAMANO_YOLO`: 960 es el valor original; 640 es más rápido pero puede perder locotes pequeños o lejanos.
TAMANO_YOLO = "960"  #@param ["960", "640"]
#@markdown `SESION_CALIBRACION`: vacío = la última calibración que hiciste en la parte 4 (o, si no hiciste ninguna, la incluida en el repositorio). Escribe un nombre para usar otra; si Colab se reinició, la trae de tu Drive.
SESION_CALIBRACION = ""  #@param {type:"string"}
import os, subprocess, sys, time
from pathlib import Path
%cd /content/deteccion-locotes
log = Path("/content/servidor.log")
# Cerrar cualquier servidor anterior (de la app o de la captura de calibración)
subprocess.run(["pkill", "-f", "yolo_segmentacion_video_ampliado|captura_webrtc"])
time.sleep(1)
sesion = SESION_CALIBRACION.strip()
if not sesion and Path("/content/sesion_activa.txt").exists():
    sesion = Path("/content/sesion_activa.txt").read_text().strip()
if sesion:
    asegurar_sesion(sesion)
print("Calibración:", sesion or "la incluida en el repositorio (calibracion_2026-10-31_02)")
servidor = subprocess.Popen([sys.executable, "-u", "fotos/yolo_segmentacion_video_ampliado.py"],
                            stdout=log.open("w"), stderr=subprocess.STDOUT,
                            env={**os.environ, "WATSOCOTE_IMG_SIZE": TAMANO_YOLO, "WATSOCOTE_SESION": sesion})
for _ in range(120):
    time.sleep(1)
    if "8766" in log.read_text() or servidor.poll() is not None:
        break
print(log.read_text())
print("✅ Servidor listo" if servidor.poll() is None else "❌ El servidor se detuvo: revisa el mensaje de arriba")

sys.path.insert(0, "/content/deteccion-locotes/fotos")
import configurar_video
print("\nEnlaces de emisión — abre cada uno EN SU CELULAR:")
for lado, url in configurar_video.enlaces().items():
    print(f"  {'Izquierda' if lado == 'izq' else 'Derecha'}: {url}")

In [ ]:
#@title Abrir la aplicación
from google.colab.output import eval_js, serve_kernel_port_as_window
url = eval_js("google.colab.kernel.proxyPort(8766)")
print("Aplicación:", url)
serve_kernel_port_as_window(8766, anchor_text="👉 Abrir WATSOCOTE en una pestaña nueva")

In [ ]:
#@title Ver el registro del servidor
print(Path("/content/servidor.log").read_text()[-4000:])

In [ ]:
#@title Copiar los datos de entrenamiento guardados a Google Drive
from google.colab import drive
import shutil
drive.mount("/content/drive")
origen = Path("/content/deteccion-locotes/fotos/dataset_carrito_pendiente")
if origen.exists():
    destino = Path("/content/drive/MyDrive/watsocote_dataset_pendiente")
    shutil.copytree(origen, destino, dirs_exist_ok=True)
    print("Copiado a", destino)
else:
    print("No hay datos guardados (activa la opción en la aplicación).")

In [ ]:
#@title Detener el servidor
servidor.terminate(); servidor.wait(10)
print("Servidor detenido.")

## 4. Calibrar las cámaras

Hay que calibrar **cada vez que muevas o cambies los soportes**, la cámara elegida o el zoom de los celulares. Sin eso, las distancias salen mal; la detección no se afecta.

**Necesitas:**
- El tablero ChArUco impreso: 8×6 cuadros de **27 mm**, marcadores de 20 mm, `DICT_4X4_50`. La primera celda genera un PDF para imprimir si no lo tienes.
- Los dos celulares **fijos en sus soportes definitivos**, transmitiendo como en la parte 3, con la misma cámara trasera y el mismo zoom que vas a usar.

**Pasos:** imprimir el tablero (opcional) → **Iniciar la captura** → tomar 20–30 pares → **Calibrar**. La calibración se guarda en una sesión nueva (no pisa la anterior) y en tu Drive, en `watsocote_calibraciones`. Después, la parte 3 la usa automáticamente.

In [ ]:
#@title 4.1 Descargar el tablero para imprimir (opcional)
#@markdown Imprímelo en A4 **a tamaño real (100 %)**, sin "Ajustar a la página". Después mide un cuadro con una regla: debe medir 27 mm. Si mide otra cosa, anota la medida para la celda 4.4.
import cv2, numpy as np
from PIL import Image
CUADRO_PX = 319                                   # 27 mm a ~300 ppp
ppp = CUADRO_PX / 27 * 25.4
tablero = cv2.aruco.CharucoBoard((8, 6), 27.0, 20.0, cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50))
img = tablero.generateImage((8 * CUADRO_PX, 6 * CUADRO_PX), marginSize=0)
hoja = Image.new("L", (round(297 / 25.4 * ppp), round(210 / 25.4 * ppp)), 255)   # A4 apaisada
hoja.paste(Image.fromarray(img), ((hoja.width - img.shape[1]) // 2, (hoja.height - img.shape[0]) // 2))
hoja.save("/content/tablero_charuco_A4.pdf", resolution=ppp)
print(f"Tablero de {8*27} × {6*27} mm en A4 apaisada.")
from google.colab import files
files.download("/content/tablero_charuco_A4.pdf")

In [ ]:
#@title 4.2 Iniciar la captura de calibración
#@markdown `NOMBRE_SESION`: vacío = se crea uno con la fecha (por ejemplo `calibracion_2026-11-21_1530`). Para agregar pares a una sesión ya empezada, escribe su nombre.
NOMBRE_SESION = ""  #@param {type:"string"}
import os, re, subprocess, sys, time
from datetime import datetime
from pathlib import Path
%cd /content/deteccion-locotes
nombre = NOMBRE_SESION.strip() or datetime.now().strftime("calibracion_%Y-%m-%d_%H%M")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", nombre):
    raise ValueError("Usa solo letras, números, '-' o '_' en el nombre de la sesión.")
if (carpeta_sesion(nombre) / "calibracion_charuco.npz").exists():
    raise ValueError(f"La sesión '{nombre}' ya está calibrada. Elige otro nombre.")
subprocess.run(["pkill", "-f", "yolo_segmentacion_video_ampliado|captura_webrtc"]); time.sleep(1)
log_captura = Path("/content/captura.log")
captura = subprocess.Popen([sys.executable, "-u", "fotos/captura_webrtc.py"], stdout=log_captura.open("w"),
                           stderr=subprocess.STDOUT, env={**os.environ, "WATSOCOTE_SESION": nombre})
for _ in range(30):
    time.sleep(1)
    if "8765" in log_captura.read_text() or captura.poll() is not None:
        break
if captura.poll() is not None:
    raise RuntimeError(log_captura.read_text())
Path("/content/sesion_en_captura.txt").write_text(nombre)
print("✅ Captura lista. Sesión:", nombre)
print("\n".join([
    "",
    "Cómo tomar las fotos:",
    "  - Pulsa 'Capturar par' con el tablero QUIETO y visible entero en las DOS cámaras.",
    "  - Muévelo entre foto y foto: cerca (~30 cm) y lejos (~1 m), a la izquierda, derecha, arriba,",
    "    abajo y en los bordes de la imagen, e inclinado hacia distintos lados (hasta ~40 grados).",
    "  - Toma entre 20 y 30 pares. Con la celda 4.3 ves cuántos van y si se detecta el tablero.",
    "  - No muevas los celulares ni toques el zoom durante ni después de la calibración.",
]))
from google.colab.output import serve_kernel_port_as_window
serve_kernel_port_as_window(8765, anchor_text="👉 Abrir la página de captura en una pestaña nueva")

In [ ]:
#@title 4.3 Ver cuántos pares van
import cv2, numpy as np, matplotlib.pyplot as plt
nombre = Path("/content/sesion_en_captura.txt").read_text().strip()
carpeta = carpeta_sesion(nombre)
izq = sorted((carpeta / "fotos_izq").glob("izq_*.png")); der = sorted((carpeta / "fotos_der").glob("der_*.png"))
print(f"Sesión {nombre}: {min(len(izq), len(der))} pares (se necesitan al menos 8; mejor 20-30)")
if izq and der:
    detector = cv2.aruco.CharucoDetector(cv2.aruco.CharucoBoard((8, 6), 27.0, 20.0,
                                         cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)))
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    for eje, ruta in zip(ejes, (izq[-1], der[-1])):
        img = cv2.imread(str(ruta)); esquinas, ids, _, _ = detector.detectBoard(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY))
        # Dibujo a mano: el formato de las esquinas cambia entre versiones de OpenCV.
        pts = np.zeros((0, 2)) if esquinas is None else np.asarray(esquinas, np.float32).reshape(-1, 2)
        n = 0 if ids is None else int(np.asarray(ids).size)
        for x, y in pts:
            cv2.circle(img, (int(x), int(y)), 6, (0, 255, 0), 2)
        eje.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); eje.axis("off")
        eje.set_title(f"{ruta.name}: {n} esquinas " + ("OK" if n >= 10 else "(mínimo 10)"))
    plt.show()

In [ ]:
#@title 4.4 Calibrar
#@markdown `SEPARACION_MM` (opcional): distancia entre las lentes de los dos celulares, medida con regla. Sirve para comprobar el resultado; 0 = no comprobar.
SEPARACION_MM = 0  #@param {type:"number"}
#@markdown `CUADRO_MM`: lado real de un cuadro del tablero impreso, medido con regla.
CUADRO_MM = 27.0  #@param {type:"number"}
import os, shutil, subprocess, sys
nombre = Path("/content/sesion_en_captura.txt").read_text().strip()
subprocess.run(["pkill", "-f", "captura_webrtc"])
r = subprocess.run([sys.executable, "fotos/calibrar_sesion_actual.py"], cwd="/content/deteccion-locotes",
                   capture_output=True, text=True,
                   env={**os.environ, "WATSOCOTE_SESION": nombre, "WATSOCOTE_BASELINE_MM": str(SEPARACION_MM),
                        "WATSOCOTE_ROTACION_GRADOS": "0", "WATSOCOTE_CUADRO_MM": str(CUADRO_MM)})
print(r.stdout[-6000:], r.stderr[-3000:])
npz = carpeta_sesion(nombre) / "calibracion_charuco.npz"
if npz.exists():
    Path("/content/sesion_activa.txt").write_text(nombre)
    montar_drive()
    destino = DRIVE_CALIBRACIONES / nombre
    shutil.copytree(carpeta_sesion(nombre), destino, dirs_exist_ok=True)
    print(f"\n✅ Calibración '{nombre}' lista y copiada a tu Drive: {destino}")
    print("Revisa arriba: RMS estéreo < 1 px y error de triangulación de ~1 mm o menos.")
    print("Ahora ve a la parte 3 y ejecuta 'Iniciar servidor': usará esta calibración.")
else:
    print("\n❌ No se pudo calibrar. Lee el mensaje de arriba: casi siempre faltan pares con el tablero visible "
          "en las dos cámaras. Ejecuta otra vez 4.2 con el MISMO nombre de sesión para agregar más pares.")